<!-- notebook-header -->
<a id="top"></a>

# 6. Trees & Tries

*LeetCode Playbook · notebook 6 of 12* · What a recursive call passes down, returns up and records; prefix trees for many words.

**In this notebook:** [Trees](#s11) · [Tries](#s12)

← [Binary Search & Sorting](05_Binary_Search_Sorting.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Heaps, Intervals & Greedy](07_Heaps_Intervals_Greedy.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s11"></a>

## Trees

> A tree recursion is a conversation. Each call gets facts from its **parent** (arguments: depth, bounds, the path so far), asks its two **children** for facts about their subtrees (return values: height, sum, "found it"), and may **record** something in a global answer on the side. Decide those three things and the code nearly writes itself.

**Reach for it when** the input is a `TreeNode` (or anything with parents and children); the answer at a node depends on its **subtrees** (height, balance, sums, diameter) or on its **ancestors** (bounds, depth, path sums); the problem says *level*, *row* or *view from the side* (BFS); or it says *BST* (inorder is sorted; compare and go one way).

**In this repo:** `trees/` (28 problems) · bank: `practice/simple/26_binary_tree_level_order_traversal.py`, `practice/simple/27_validate_binary_search_tree.py`, `practice/simple/28_kth_smallest_element_in_a_bst.py`, `practice/simple/29_lowest_common_ancestor_of_a_bst.py`, `practice/simple/30_diameter_of_binary_tree.py` · basics: `practice/simple/basics/trees/` (traversals recursive and iterative, BFS level order and height, BST insert/search/delete, balanced, LCA, serialize/deserialize).

### The picture

Information moves in two directions. **Down**, as arguments: what the ancestors know. **Up**, as return values: what a subtree knows about itself.

```text
tree = [3, 9, 20, None, None, 15, 7]

                  arguments DOWN (depth)        return values UP (height)

        3         depth 0                       3 returns 1 + max(1, 2) = 3
      /   \
     9     20     depth 1                       9 returns 1      20 returns 1 + max(1, 1) = 2
          /  \
        15    7   depth 2                       15 returns 1     7 returns 1
```

Every call is the same small box. It receives arguments from its parent, gets one return value from each child, sends one value up, and may write to a global answer that nobody returns:

```text
                         parent
             arguments  |      ^  return value
       (depth, bounds,  |      |  (height, sum, found node)
        path so far)    v      |
                    +--------------+
                    |     node     |------>  RECORD into a global answer
                    +--------------+         (best, a result list)
                     |  ^      |  ^
                args v  | ret  v  | ret
                    left        right
```

Why it is fast: the brute force usually recomputes a fact about a subtree once for every ancestor (calling `height()` from each node in Balanced or Diameter is O(n²) on a chain), or re-walks the path from the root at every node (Count Good Nodes rescanning the path is O(n·h)). A bottom-up pass computes each subtree fact exactly once and hands it to the parent; a top-down pass hands each node a summary of its ancestors (a running max, a window, a remaining sum) in O(1). Either way every node is visited once: O(n) time, O(h) space for the recursion (h = height).

### From idea to code

**The idea in one sentence:** *finish the sentence "f(node) returns ___ about node's subtree, given ___ from its parent, and records ___", then write the base case for `None`, recurse into the children, and combine.*

| Decision | Tree-recursion answer |
|---|---|
| **State**: what must I remember? | the arguments (facts from above: depth, bounds, remaining sum, the path), the return value (a fact about this subtree), and maybe one global (`best`, a result list) |
| **Definition**: what exactly does each variable mean? | one sentence per function, as a comment: `height(node)` = number of nodes on the longest downward path from `node`; 0 for `None` |
| **Invariant**: what is true at the end of every step? | trust the recursion: when a child's call returns, its answer is correct for that child's *whole* subtree |
| **Step**: how does one node change the state? | base case → recurse into the children (with updated arguments) → combine their two answers with `node.val` |
| **Record**: when is the answer updated? | when the problem's answer is *not* what the parent needs (a path that bends at this node, a finished level, a matching path): write it to the global inside the call |
| **Init**: starting values | the answer for an empty tree (`0`, `True`, `None`), the root's arguments (`-inf, inf`, depth 0), and `best` = the worst possible (`0` for lengths, `-math.inf` for sums) |
| **Return**: what comes back? | `f(root)` or the recorded global, translated if needed (`height(root) != -1`) |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "an empty tree" | `if node is None: return 0` (or `True`, or `None`: the neutral answer) |
| "a leaf" | `if node.left is None and node.right is None:` |
| "ask both children" | `left, right = f(node.left), f(node.right)` |
| "tell my children what I know" | `f(node.left, depth + 1)`, `valid(node.left, low, node.val)` |
| "my answer from theirs" | `return 1 + max(left, right)` |
| "remember the best seen anywhere" | `nonlocal best`, then `best = max(best, left + right)` |
| "the same node" | `node is p` (not `node.val == p.val`) |
| "one level at a time" | `for _ in range(len(queue)):` |
| "the sorted order of a BST" | inorder: left subtree, node, right subtree |

**The key question: what do I return, what do I record?** If the parent needs exactly what the problem asks for, just return it. If not, return what the parent needs and record the answer on the side. For every problem in this section:

| Problem | Down (from the parent) | Up (return value) | Recorded globally |
|---|---|---|---|
| Max depth (104) | – | height | – |
| Balanced (110) | – | height, or -1 = "unbalanced below" | – |
| Diameter (543) | – | height (one arm) | `best = max(best, left + right)` |
| Max path sum (124) | – | best one-arm sum going down (the parent clips a negative one to 0) | `best = max(best, val + left + right)` |
| Validate BST (98) | open window `(low, high)` | `True` / `False` | – |
| Count good nodes (1448) | largest value on the path | number of good nodes in this subtree | – |
| Path Sum II (113) | remaining sum, the shared `path` | – | copies of the matching paths |
| Right side view (199) | depth | – | the first value seen at each depth |
| LCA (236) | – | `p`, `q`, the LCA, or `None` | – |
| Cameras (968) | – | needs / covered / camera | the number of cameras |

First the node, and two helpers: build a tree from LeetCode's level-order list (`None` = no child) and turn a tree back into that list.

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def build_tree(values):                      # LeetCode level order, None = no child
    nodes = [None if v is None else TreeNode(v) for v in values]
    kids = iter(nodes[1:])
    for node in nodes:
        if node:                             # each real node takes the next two slots
            node.left, node.right = next(kids, None), next(kids, None)
    return nodes[0] if nodes else None


def tree_to_list(root):                      # the inverse: tree -> LeetCode list
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        out.append(node.val if node else None)
        if node:
            queue.extend([node.left, node.right])
    while out and out[-1] is None:           # LeetCode trims the trailing Nones
        out.pop()
    return out


root = build_tree([3, 9, 20, None, None, 15, 7])
print(root.val, root.left.val, root.right.left.val)   # 3 9 15
print(tree_to_list(root))                             # [3, 9, 20, None, None, 15, 7]

**Try it**
- Build `[1, None, 2, 3]` and print `root.right.left.val`: 3. The `None` fills only 1's left slot, so 2 is 1's right child and 3 hangs to the left of 2.
- `build_tree([])` is `None` and `tree_to_list(None)` is `[]`: the empty tree round-trips.
- Delete the trimming loop in `tree_to_list`: the example prints `[3, 9, 20, None, None, 15, 7, None, None, None, None]`, two `None`s for each of the two bottom leaves.

The three shapes of tree recursion. Bottom-up (`max_depth`): children answer first, the parent combines. Top-down (`is_valid_bst`): the parent passes a constraint down. Both at once (`diameter`): return one thing, record another. The order of the lines is a decision here too: a bottom-up call can only combine after both children have answered (post-order), and in `diameter` the RECORD sits between the children's answers and the RETURN, because it needs both arms while the parent gets only one.

In [ ]:
def max_depth(node):                          # bottom-up: the answer comes UP
    if node is None:                          # INIT: an empty tree has depth 0
        return 0
    left = max_depth(node.left)               # STEP: ask both children first ...
    right = max_depth(node.right)
    return 1 + max(left, right)               # RETURN: ... then combine (post-order)


def is_valid_bst(root):                       # top-down: the constraint goes DOWN
    def valid(node, low, high):               # STATE: every value here must be in (low, high)
        if node is None:
            return True                       # an empty subtree breaks nothing
        if not low < node.val < high:
            return False                      # one node outside its window decides it
        return (valid(node.left, low, node.val)         # STEP: going left caps high
                and valid(node.right, node.val, high))  # going right raises low
    return valid(root, -math.inf, math.inf)   # INIT: the root may be anything


def diameter(root):                           # both: RETURN one thing, RECORD another
    best = 0                                  # STATE: longest path seen anywhere, in edges
    def height(node):
        nonlocal best                         # we reassign best, so nonlocal is required
        if node is None:
            return 0                          # INIT: an empty tree has height 0
        left, right = height(node.left), height(node.right)   # STEP: both children answer first
        best = max(best, left + right)        # RECORD: the path that bends at node
        return 1 + max(left, right)           # RETURN: only one arm can continue upward
    height(root)
    return best                               # RETURN: the recorded answer, not height(root)


tree = build_tree([3, 9, 20, None, None, 15, 7])
print(max_depth(tree), is_valid_bst(tree), diameter(tree))          # 3 False 3
print(is_valid_bst(build_tree([5, 1, 7, None, None, 6, 8])))        # True
print(diameter(build_tree([1, 2, None, 3, 4, 5, None, None, 6])))   # 4  (bends at 2, not at the root)

**Try it**
- In `diameter`, return `1 + left + right` (both arms) instead of `1 + max(left, right)`: the last example says 5 instead of 4, because the parent now extends a "path" that already forks.
- Delete `nonlocal best`: `UnboundLocalError`. Assigning to `best` inside `height` makes it a new local name. (Appending to a list needs no `nonlocal`: you change the list, you don't rebind the name.)
- Replace the window with a parent-only check, `valid(node.left, -math.inf, node.val)` and `valid(node.right, node.val, math.inf)`, then try `is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7]))`: `True`, although the 3 sits in 5's right subtree.
- Change `low < node.val < high` to `low <= node.val <= high`: `is_valid_bst(build_tree([2, 2]))` becomes `True`. A BST here has no duplicates.

**DFS orders.** Preorder, inorder and postorder are the same walk; they differ only in *when* you write the node down. Postorder is the bottom-up order (children first), preorder is the top-down order (parent first), and the inorder of a BST is sorted.

```text
            1            preorder  (node, left, right):  1 2 4 5 3 6    node BEFORE its children
          /   \          inorder   (left, node, right):  4 2 5 1 3 6    node BETWEEN them
         2     3         postorder (left, right, node):  4 5 2 6 3 1    node AFTER them
        / \     \
       4   5     6
```

The recursion keeps a hidden stack of "nodes waiting for me to come back". You can keep that stack yourself, which avoids Python's recursion limit and lets you pause the walk (the BST iterator, kth smallest):

In [ ]:
def three_orders(root):                       # one walk, three moments to write the node
    pre, ino, post = [], [], []
    def walk(node):
        if node is None:
            return
        pre.append(node.val)                  # before the children
        walk(node.left)
        ino.append(node.val)                  # between them
        walk(node.right)
        post.append(node.val)                 # after them
    walk(root)
    return pre, ino, post


def preorder_iterative(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)                  # visit when popped
        if node.right:
            stack.append(node.right)          # push right first ...
        if node.left:
            stack.append(node.left)           # ... so left is popped first
    return out


def inorder_iterative(root):
    out, stack, node = [], [], root
    while node or stack:                      # both: a right subtree may still be pending
        while node:                           # slide down the left edge, leaving breadcrumbs
            stack.append(node)
            node = node.left
        node = stack.pop()                    # its left side is done: visit it
        out.append(node.val)
        node = node.right                     # then do its right subtree the same way
    return out


def postorder_iterative(root):                # node-right-left, then reversed
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return out[::-1]                          # reversed node-right-left = left-right-node


t = build_tree([1, 2, 3, 4, 5, None, 6])
pre, ino, post = three_orders(t)
print(pre, ino, post)                         # [1, 2, 4, 5, 3, 6] [4, 2, 5, 1, 3, 6] [4, 5, 2, 6, 3, 1]
print(preorder_iterative(t) == pre, inorder_iterative(t) == ino, postorder_iterative(t) == post)   # True True True

**Try it**
- In `inorder_iterative`, move `out.append(node.val)` up into the `while node:` loop (record when you push): you get `[1, 2, 4, 5, 3, 6]`, the preorder. Recording on arrival is preorder; recording once the left side is done is inorder.
- Print `[n.val for n in stack]` right after the inner `while node:` loop: `[1, 2, 4]`, `[1, 2]`, `[1, 5]`, `[1]`, `[3]`, `[6]`. The stack only ever holds part of one root-to-leaf path, so it is O(h).
- In `preorder_iterative`, push left before right: `[1, 3, 6, 2, 5, 4]`, the mirror image. A stack pops the last thing pushed.
- Remove the `[::-1]` from `postorder_iterative`: `[1, 3, 6, 2, 5, 4]` again (node, right, left). Postorder is that list read backwards.

**BFS by level.** A queue visits nodes in order of depth. The one trick for grouping: at the start of a round the queue holds exactly one level, so `len(queue)` is that level's size.

```text
          3            round 1: queue = [3]       -> level [3]
         / \           round 2: queue = [9, 20]   -> level [9, 20]
        9   20         round 3: queue = [15, 7]   -> level [15, 7]
           /  \
          15   7       each popped node puts its children at the BACK, behind the current level
```

In [ ]:
def level_order(root):
    levels, queue = [], deque([root] if root else [])   # STATE + INIT: queue = one level, left to right
    while queue:
        level = []
        for _ in range(len(queue)):           # snapshot: exactly the nodes of this level
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)       # STEP: children wait behind this level
            if node.right:
                queue.append(node.right)
        levels.append(level)                  # RECORD: one finished level
    return levels                             # RETURN


def right_side_view(root):                    # top-down DFS: the depth goes down
    view = []
    def dfs(node, depth):
        if node is None:
            return
        if depth == len(view):                # the first node to reach this depth
            view.append(node.val)
        dfs(node.right, depth + 1)            # right first, so it claims the level
        dfs(node.left, depth + 1)
    dfs(root, 0)
    return view


print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))    # [[3], [9, 20], [15, 7]]
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))  # [1, 3, 4]
print(right_side_view(build_tree([1, 2, 3, 4])))                 # [1, 3, 4]  (4 is visible: nothing to its right)

**Try it**
- Replace `for _ in range(len(queue)):` with `while queue:`: the first tree becomes one level, `[[3, 9, 20, 15, 7]]`. The children joined the round that was still running.
- Print `len(queue)` at the top of each round of `level_order`: 1, 2, 2, the level sizes.
- In `right_side_view`, recurse left before right: `[1, 2, 3, None, 5, None, 4]` now gives `[1, 2, 5]`, the view from the *left*.
- A second way to the right side view: `[level[-1] for level in level_order(r)]`. Check that it agrees on both trees.

### Watch it work

Two traces. The first shows the bottom-up order: a node only "leaves" after both children have left, carrying the record (`best`) and the return value separately. The second shows top-down windows shrinking on the way down.

In [ ]:
def trace_diameter(root):
    best = 0
    def height(node, depth):
        nonlocal best
        if node is None:
            return 0
        pad = "    " * depth
        print(f"{pad}enter {node.val}")
        left, right = height(node.left, depth + 1), height(node.right, depth + 1)
        best = max(best, left + right)
        print(f"{pad}leave {node.val}: arms {left} + {right}, best = {best}, return {1 + max(left, right)}")
        return 1 + max(left, right)
    height(root, 0)


def trace_windows(root):
    def valid(node, low, high, depth):
        if node is None:
            return True
        ok = low < node.val < high
        print(f"{'    ' * depth}{node.val} must be in ({low}, {high}): {'ok' if ok else 'BROKEN'}")
        return ok and valid(node.left, low, node.val, depth + 1) and valid(node.right, node.val, high, depth + 1)
    return valid(root, -math.inf, math.inf, 0)


trace_diameter(build_tree([1, 2, 3, 4, 5]))
print(trace_windows(build_tree([5, 1, 7, None, None, 4, 8])))

**Try it**
- Run `trace_diameter(build_tree([1, 2, None, 3, 4, 5, None, None, 6]))` and find the line where `best` becomes 4: it is when node 2 leaves. The root only passes the height up.
- Each "leave" line comes after the "leave" lines of both children. That is post-order, and it is why a node can trust both arms.
- In `trace_windows`, delete `ok and` from the return line: the 4 is still printed as BROKEN, yet the answer becomes `True`. A node's own check must be part of what it returns.
- Run `trace_windows(build_tree([5, 1, 7, None, None, 6, 8]))`: every line is ok, and 6's window is (5, 7).

### Where it goes wrong

1. **Returning what you should record.** In Diameter and Max Path Sum the parent can extend only one arm: return `1 + max(left, right)` (or `val + max(left, right)`) and *record* `left + right`. Returning both arms counts paths that fork (`[1, 2, None, 3, 4, 5, None, None, 6]` gives 5 instead of 4).
2. **`UnboundLocalError` on the global.** Assigning `best = ...` inside a nested function makes `best` local. Add `nonlocal best` (appending to a list needs no `nonlocal`).
3. **Checking only parent and child in a BST.** `[5, 4, 6, None, None, 3, 7]` passes a parent-only check. Pass the window `(low, high)` down instead.
4. **The wrong neutral value.** Max Path Sum with `best = 0` answers 0 for `[-3]` (should be -3): start at `-math.inf`. In Balanced, an empty subtree has height 0; -1 means "unbalanced".
5. **A leaf is not `None`.** "Root-to-leaf" checks belong at nodes with no children. Testing `remaining == 0` when you reach `None` makes `[1, 2]` with target 1 succeed through 1's empty right side.
6. **Sharing one mutable path.** `paths.append(path)` stores the *same* list again and again; append `path[:]`, and `path.pop()` after the recursive calls.
7. **Mixing up levels in BFS.** Read `len(queue)` once per round (the `range(...)` does that); a `while queue:` inner loop drains several levels into one.
8. **Values instead of nodes.** LCA, cousins and "is it the same node" compare identity: `node is p`.
9. **Deep recursion.** A chain of 10⁴ nodes exceeds Python's default recursion limit (1000). Use an explicit stack, or raise the limit with `sys.setrecursionlimit`.

### Edge cases to say out loud

Empty tree (`None`) · a single node · a chain (height = n, recursion depth!) · all values negative · duplicates (BST strictness; compare nodes by identity) · p is an ancestor of q · the answer does not pass through the root · values at the integer limits (use `math.inf` bounds, not a sentinel like `2**31 - 1` that a real value can equal).

In [ ]:
assert max_depth(None) == 0 and diameter(None) == 0 and is_valid_bst(None)
assert max_depth(build_tree([1])) == 1 and diameter(build_tree([1])) == 0
chain = build_tree([1, None, 2, None, 3, None, 4])       # every node has only a right child
assert max_depth(chain) == 4 and diameter(chain) == 3 and is_valid_bst(chain)
assert not is_valid_bst(build_tree([2, 2]))              # duplicates break the strict order
assert is_valid_bst(build_tree([2 ** 31 - 1]))           # math.inf bounds never collide with values
assert level_order(None) == [] and right_side_view(None) == []
assert inorder_iterative(chain) == [1, 2, 3, 4] == preorder_iterative(chain)
print("edge cases pass")

**Try it**
- Predict, then check: `is_valid_bst(build_tree([1, None, 1]))` is `False`. An equal value may not sit on the right either.
- Build a 10 000-node chain: `deep = build_tree([v for i in range(10_000) for v in (i, None)])`. `max_depth(deep)` raises `RecursionError`, while `len(inorder_iterative(deep))` is 10000.
- What is the diameter of the "V" `build_tree([1, 2, 3, 4, None, None, 5])`? Write the assert first: 4 (the path 4-2-1-3-5).

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Bottom-up value** | return a fact about the subtree; combine the two children | 104, 226, 110 |
| **Return ≠ record** | return one arm; record the path that bends here | 543, 124 |
| **Sentinel or state return** | return -1 or a small state instead of a plain number | 110, 968 |
| **Top-down arguments** | pass a running max, a window, a remaining sum, a depth, a parent | 1448, 98, 112, 404, 993, 199 |
| **Path backtracking** | append on the way down, pop on the way up, copy at a matching leaf | 113 |
| **Found below me** | return `p`/`q`/LCA/`None`; a hit from both sides means "it's me" | 236 |
| **BST: compare and go one way** | a single O(h) descent, no recursion needed | 235 (and BST insert/search) |
| **Inorder = sorted** | iterative inorder with a counter or a `prev` node | 94, 230, 99, 272 |
| **BFS by level** | a queue plus the `len(queue)` snapshot | 102, 199 |
| **Coordinates** | DFS carrying `(row, col)`, bucket by column, sort | 987 |
| **Build or encode a tree** | preorder + inorder index map; preorder with `#` for `None`; a stack indexed by depth | 105, 297, 428, 1028, 572 |
| **Two trees at once** | recurse on pairs `(a, b)` | 100, 101, 572 |

**Return one arm, record the bend (124), and smuggle a failure up (110).** Max Path Sum has the skeleton of Diameter, but the arms are sums, and a negative arm is dropped (clipped to 0) because a path doesn't have to use it. Balanced shows a different trick: send "this subtree already failed" up the same return channel, as -1.

In [ ]:
def max_path_sum(root):
    best = -math.inf                          # not 0: an all-negative tree still has an answer
    def gain(node):                           # best sum going DOWN from node along one side
        nonlocal best
        if node is None:
            return 0
        left = max(gain(node.left), 0)        # a negative arm is simply left out
        right = max(gain(node.right), 0)
        best = max(best, node.val + left + right)   # RECORD: the path that bends at node
        return node.val + max(left, right)    # RETURN: only one arm continues up
    gain(root)
    return best


def is_balanced(root):
    def height(node):                         # the true height, or -1 = "unbalanced below"
        if node is None:
            return 0
        left, right = height(node.left), height(node.right)
        if left == -1 or right == -1 or abs(left - right) > 1:
            return -1                         # the failure travels up as -1
        return 1 + max(left, right)
    return height(root) != -1


print(max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])), max_path_sum(build_tree([-3])))   # 42 -3
print(is_balanced(build_tree([3, 9, 20, None, None, 15, 7])), is_balanced(build_tree([1, 2, None, 3, None, 4])))   # True False

**Try it**
- Remove both `max(..., 0)` clips and try `max_path_sum(build_tree([2, -1]))`: 1 instead of 2. Without the clip the path is forced to take the -1 arm.
- Start with `best = 0`: `[-3]` now answers 0, the sum of a path that doesn't exist.
- In `is_balanced`, print `node.val, left, right` just before the `if`, for the second tree (a chain 1-2-3-4 going left): node 2 is the first to see arms 2 and 0 and returns -1; node 1 prints `1 -1 0` and just passes the -1 up.

**Lowest common ancestor (236, 235).** Ask every subtree one question, "which of p and q do you contain?", and let the answer be a single node: `None` (neither), `p` or `q` (that one), or the LCA itself (both, already resolved below). The first node that hears a non-`None` answer from *both* sides is where the two paths split. In a BST you don't need to search at all: compare values and walk down one path.

```text
              3               lca(5, 1): 3 hears "5" from the left and "1" from the right:
            /   \                        both sides answered, so 3 is the split point
           5     1
          / \   / \           lca(5, 4): the call on 5 returns 5 at once (it IS p);
         6   2 0   8                     the right side reports nothing, so 3 passes 5 up
            / \
           7   4
```

In [ ]:
def lca(root, p, q):                          # returns p, q, their LCA, or None for this subtree
    if root is None or root is p or root is q:
        return root                           # found one (or nothing): report it up
    left = lca(root.left, p, q)
    right = lca(root.right, p, q)
    if left and right:
        return root                           # one target on each side: I am the split
    return left or right                      # pass up whatever was found


def lca_bst(root, p, q):                      # BST: compare values, walk ONE path down
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    node = root
    while node:
        if hi < node.val:
            node = node.left                  # both targets are smaller
        elif lo > node.val:
            node = node.right                 # both are larger
        else:
            return node                       # lo <= node.val <= hi: they split here


def find_node(root, val):                     # helper: the node holding val
    if root is None or root.val == val:
        return root
    return find_node(root.left, val) or find_node(root.right, val)


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
print(lca(t, find_node(t, 5), find_node(t, 1)).val, lca(t, find_node(t, 5), find_node(t, 4)).val)   # 3 5
b = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
print(lca_bst(b, find_node(b, 2), find_node(b, 8)).val, lca_bst(b, find_node(b, 3), find_node(b, 5)).val)   # 6 4

**Try it**
- Add `print(root.val)` right after the base-case `if` in `lca`, rerun the cell, then call `lca(t, find_node(t, 5), find_node(t, 4))`: only 3, 1, 0 and 8 are printed. Nothing below 5 is visited: the call on 5 returns at once, and since nobody else reports anything, 5 travels up as the answer.
- Compare values instead of nodes (`root.val in (p.val, q.val)` in the base case), then try `t2 = build_tree([1, 2, 3, None, None, 2])` with `p, q = t2.right.left, t2.right`: you get 1 instead of 3, because the *other* 2 answered for p.
- In `lca_bst`, change `hi < node.val` to `hi <= node.val` and ask for 2 and 0: you get 0 instead of 2. When a target *is* the node, the node is the answer (a node counts as its own ancestor).

**A path shared by the whole recursion (113, 1448).** Carry the remaining sum down as an argument, and keep ONE list for the current path: append on the way down, pop on the way back, and copy it only at a matching leaf. Count Good Nodes is the same top-down shape with a running maximum instead of a path.

In [ ]:
def path_sum_all(root, target):
    paths, path = [], []
    def dfs(node, remaining):
        if node is None:
            return
        path.append(node.val)                 # going down: extend the path
        remaining -= node.val
        if node.left is None and node.right is None:
            if remaining == 0:
                paths.append(path[:])         # RECORD a COPY: path keeps changing
        else:
            dfs(node.left, remaining)
            dfs(node.right, remaining)
        path.pop()                            # coming back up: undo
    dfs(root, target)
    return paths


def good_nodes(root):
    def dfs(node, path_max):                  # path_max = the largest value above node
        if node is None:
            return 0
        good = 1 if node.val >= path_max else 0
        path_max = max(path_max, node.val)
        return good + dfs(node.left, path_max) + dfs(node.right, path_max)
    return dfs(root, -math.inf)


t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
print(path_sum_all(t, 22))                            # [[5, 4, 11, 2], [5, 8, 4, 5]]
print(good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])))   # 4

**Try it**
- Replace `paths.append(path[:])` with `paths.append(path)`: `[[], []]`. Both entries are the same list object, emptied by the pops on the way back.
- Delete `path.pop()`: the sums are still checked correctly (`remaining` travels as an argument), but the paths keep stale nodes: the first answer becomes `[5, 4, 11, 7, 2]`.
- Call `path_sum_all(t, 9)`: `[]`, even though 5 + 4 = 9, because that 4 is not a leaf.
- In `good_nodes`, change `>=` to `>`: 3 instead of 4. The lower 3 only *ties* the largest value above it, and ties count as good.

**Inorder is sorted (230, 99).** Walk a BST inorder and you get its values in increasing order, one at a time. Written as a generator, the explicit-stack inorder becomes a lazy iterator: `yield` hands out one node and pauses the loop right there until the caller asks for the next. Kth Smallest stops after k nodes. Recover BST compares each node with the previous one: two swapped values make the sorted walk dip once (if they were neighbours) or twice (if far apart).

```text
 sorted inorder:   1  2  3  4  5  6
 2 and 5 swapped:  1  5  3  4  2  6        dip 1: 5 > 3        dip 2: 4 > 2
                                           first = 5, the bigger value of the FIRST dip
                                           second = 2, the smaller value of the LAST dip
 neighbours swapped (1 3 2 4) make one dip, and the same rule picks 3 and 2 from it
```

In [ ]:
def inorder_nodes(root):                      # yields the nodes in sorted order, lazily
    stack, node = [], root
    while node or stack:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        yield node
        node = node.right


def kth_smallest(root, k):
    for i, node in enumerate(inorder_nodes(root), 1):
        if i == k:
            return node.val                   # stop early: the rest is never visited


def recover_bst(root):
    first = second = prev = None
    for node in inorder_nodes(root):
        if prev and prev.val > node.val:      # a dip in what should be increasing
            if first is None:
                first = prev                  # the bigger value of the FIRST dip
            second = node                     # the smaller value of the LAST dip
        prev = node
    first.val, second.val = second.val, first.val


print(kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))   # 3
t = build_tree([3, 1, 4, None, None, 2])
recover_bst(t)
print(tree_to_list(t))                        # [2, 1, 4, None, None, 3]

**Try it**
- Print `[n.val for n in inorder_nodes(t)]` before and after `recover_bst(t)` (rebuild `t` first): `[1, 3, 2, 4]`, then `[1, 2, 3, 4]`.
- Make the loop stop at the first dip (`break` right after `second = node`) and repair `build_tree([4, 5, 6, 1, 3, 2])`, where 2 and 5 were swapped far apart: it swaps 5 with 3 instead, and the inorder becomes `[1, 3, 5, 4, 2, 6]`. The full version gives `[1, 2, 3, 4, 5, 6]`.
- Add `print(node.val)` after `stack.append(node)` and call `kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 1)`: only 5, 3, 2, 1 are ever pushed. The walk stops as soon as the answer is known: O(h + k).

**Build a tree from its traversals (105) and serialize it (297).** Preorder says *who* the root is (the first unused value); inorder says *how many* nodes are on its left (everything before the root). A dict finds the root's inorder position in O(1), and one iterator walks preorder, because building left-then-right uses preorder in exactly its own order. To *store* a tree, preorder alone is ambiguous, but preorder with `#` for every missing child is complete: each `#` says "this branch stops here", so the reader can replay the walk.

```text
 preorder = [3 | 9 | 20, 15, 7]      the next unused preorder value is the root: 3
 inorder  = [9 | 3 | 15, 20, 7]      left of 3 in inorder: [9]          -> the left subtree
             left    right           right of 3: [15, 20, 7]            -> the right subtree

 serialize [1, 2, 3, None, None, 4, 5]:   1,2,#,#,3,4,#,#,5,#,#
 a value = "make a node, read its left subtree, then its right";  # = "nothing here"
```

In [ ]:
def build_from_pre_in(preorder, inorder):
    where = {v: i for i, v in enumerate(inorder)}     # value -> inorder index (values unique)
    next_root = iter(preorder)                        # roots come out in preorder order
    def make(lo, hi):                                 # builds the subtree that owns inorder[lo:hi]
        if lo >= hi:
            return None
        root = TreeNode(next(next_root))
        mid = where[root.val]
        root.left = make(lo, mid)                     # left FIRST: it uses the next preorder values
        root.right = make(mid + 1, hi)
        return root
    return make(0, len(inorder))


def serialize(root):
    tokens = []
    def walk(node):
        if node is None:
            tokens.append("#")                        # a branch ends here
            return
        tokens.append(str(node.val))
        walk(node.left)
        walk(node.right)
    walk(root)
    return ",".join(tokens)


def deserialize(data):
    tokens = iter(data.split(","))                    # one cursor shared by every call
    def read():
        token = next(tokens)
        if token == "#":
            return None
        node = TreeNode(int(token))
        node.left = read()                            # consumes exactly the left subtree's tokens
        node.right = read()
        return node
    return read()


print(tree_to_list(build_from_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))   # [3, 9, 20, None, None, 15, 7]
s = serialize(build_tree([1, 2, 3, None, None, 4, 5]))
print(s, tree_to_list(deserialize(s)))        # 1,2,#,#,3,4,#,#,5,#,# [1, 2, 3, None, None, 4, 5]

**Try it**
- Remove the `tokens.append("#")` line: `serialize(build_tree([1, 2]))` and `serialize(build_tree([1, None, 2]))` both give `"1,2"`. Without the markers, preorder can't tell a left child from a right one (and the cell's last line now fails with `StopIteration`: the reader runs out of tokens).
- `serialize(None)` is `"#"` and `deserialize("#")` is `None`: the empty tree round-trips too.
- Build from `preorder=[1, 2]` with `inorder=[2, 1]`, then with `inorder=[1, 2]`: `[1, 2]` versus `[1, None, 2]`. Same preorder; inorder decides the side.
- Swap the two recursive lines in `make` (right subtree first): the build hands preorder values to the wrong subtrees and stops with `StopIteration`, asking for more roots than there are.

**Coordinates (987).** Give every node a position on graph paper: the root at (row 0, col 0), a left child at (row + 1, col − 1), a right child at (row + 1, col + 1). Once each node carries its coordinates, the tree shape no longer matters: bucket by column, then sort each bucket by (row, value).

```text
                 1 (0,0)                  column 0 holds 1, 6 and 5; 6 and 5 share the
              /         \                 cell (2, 0), so the smaller value comes first:
         2 (1,-1)      3 (1,1)            [1, 5, 6]
         /    \        /    \
   4 (2,-2) 6 (2,0) 5 (2,0) 7 (2,2)
```

In [ ]:
def vertical_order(root):
    columns = defaultdict(list)               # col -> [(row, val), ...]
    def dfs(node, row, col):
        if node is None:
            return
        columns[col].append((row, node.val))
        dfs(node.left, row + 1, col - 1)
        dfs(node.right, row + 1, col + 1)
    dfs(root, 0, 0)
    return [[val for _, val in sorted(columns[c])] for c in sorted(columns)]


print(vertical_order(build_tree([3, 9, 20, None, None, 15, 7])))   # [[9], [3, 15], [20], [7]]
print(vertical_order(build_tree([1, 2, 3, 4, 6, 5, 7])))           # [[4], [2], [1, 5, 6], [3], [7]]

**Try it**
- Sort each column by value only (`sorted(columns[c], key=lambda rv: rv[1])`) and try `build_tree([5, 1, 9, None, 3])`: column 0 becomes `[3, 5]`, but 5 is above 3. The row decides first; the value only breaks ties.
- Print `dict(columns)` before the `return` for the second tree: column 0 holds `(0, 1), (2, 6), (2, 5)`. 6 and 5 share a grid cell, so the sort uses their values.
- Visit the right child before the left: the output doesn't change. After the sort, the order of the visit no longer matters.

**A state per node (968).** Sometimes one number is not enough, and each subtree reports one of three states to its parent. Be lazy: a node takes a camera only when a child is uncovered. A leaf never takes one, because a camera on its parent covers everything the leaf's camera would (the leaf and the parent), plus the parent's other neighbours.

```text
 states a child can report:  "needs"   dark, nobody covers me      None reports "covered",
                             "covered" lit by a camera below       so a leaf reports "needs",
                             "camera"  I have a camera             and its parent must act

        0       covered  (its child has a camera)
       /
      0         camera   (a child needs one)                   [0, 0, None, 0, 0] -> 1 camera
     / \
    0   0       needs    (leaves: nothing below lights them)
```

In [ ]:
def min_cameras(root):
    NEEDS, COVERED, CAMERA = "needs", "covered", "camera"
    cameras = 0
    def state(node):
        nonlocal cameras
        if node is None:
            return COVERED                    # nothing to watch, never asks for help
        left, right = state(node.left), state(node.right)
        if left == NEEDS or right == NEEDS:
            cameras += 1                      # RECORD: forced by a dark child
            return CAMERA
        if left == CAMERA or right == CAMERA:
            return COVERED                    # lit from below
        return NEEDS                          # children are fine, but nobody lights me
    if state(root) == NEEDS:
        cameras += 1                          # the root has no parent to ask
    return cameras


print(min_cameras(build_tree([0, 0, None, 0, 0])))                         # 1
print(min_cameras(build_tree([0, 0, None, 0, None, 0, None, None, 0])))    # 2  (a chain of 5)
print(min_cameras(build_tree([0])))                                        # 1

**Try it**
- Make `None` return `NEEDS`: the first tree now takes 3 cameras instead of 1, one on every leaf (and one on the root).
- Delete the final root check: `min_cameras(build_tree([0]))` answers 0 for a tree that needs a camera.
- Add `print(left, right)` after the line that asks the children, and run the chain of 5: from the bottom up you see `covered covered` (the leaf), `covered needs` (camera!), `camera covered`, `covered covered`, `needs covered` (camera!). The pattern repeats every three levels.

### Say it in the interview

> "The brute force recomputes the height of every subtree from each of its ancestors, which is O(n²) on a skewed tree. Instead I'll do one post-order pass: `height(node)` returns its subtree's height to the parent, and while both children's heights are in hand I record `left + right`, the longest path that bends at this node. Every node is visited once: O(n) time and O(h) space for the recursion."

Before typing, say the contract out loud: *"this function returns ___ for its subtree, receives ___ from its parent, and records ___."* Then point at the base case (what an empty tree returns), the line that combines the children, and the line that records. If the tree can be a 10⁴-node chain, mention the recursion limit and offer the explicit-stack version.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Balanced Binary Tree | `trees/balanced_binary_tree.py` | return the height, or -1 as soon as any subtree is unbalanced; the -1 travels up |
| Binary Tree Cameras | `trees/binary_tree_cameras.py` | post-order states needs/covered/camera; `None` is covered; camera only when a child needs one; check the root |
| Binary Tree Inorder Traversal | `trees/binary_tree_inorder_traversal.py` | the stack holds the left edge: push lefts, pop = visit, go right; loop while node or stack |
| Binary Tree Level Order Traversal | `trees/binary_tree_level_order_traversal.py` · `practice/simple/26_binary_tree_level_order_traversal.py` | BFS; `len(queue)` at the start of a round is the size of one level |
| Binary Tree Maximum Path Sum | `trees/binary_tree_maximum_path_sum.py` | record val + left + right; return val + max(left, right); clip negative arms to 0; best starts at -inf |
| Binary Tree Right Side View | `trees/binary_tree_right_side_view.py` | right-first DFS with depth; record when depth == len(view) |
| Closest Binary Search Tree Value II | `trees/closest_binary_search_tree_value_ii.py` | two lazy inorder stacks (values ≤ target, values > target); merge k times by distance |
| Construct Binary Tree from Preorder and Inorder Traversal | `trees/construct_binary_tree_from_preorder_and_inorder_traversal.py` | preorder gives the root, an inorder index map splits left/right; build left first with one preorder pointer |
| Count Good Nodes in Binary Tree | `trees/count_good_nodes_in_binary_tree.py` | pass the path maximum down; good iff val ≥ path max |
| Cousins in Binary Tree | `trees/cousins_in_binary_tree.py` | carry (parent, depth) down; cousins = same depth, different parent |
| Diameter of Binary Tree | `trees/diameter_of_binary_tree.py` · `practice/simple/30_diameter_of_binary_tree.py` | return the height; record left + right (edges) at every node |
| Invert Binary Tree | `trees/invert_binary_tree.py` | swap the two children at every node (tuple swap); any visiting order works |
| Kth Smallest Element in a BST | `trees/kth_smallest_element_in_a_bst.py` · `practice/simple/28_kth_smallest_element_in_a_bst.py` | inorder is sorted: iterative inorder, stop at the k-th pop |
| Lowest Common Ancestor of a Binary Tree | `trees/lowest_common_ancestor_of_a_binary_tree.py` | return p/q/LCA/None per subtree; hits from both sides make this node the LCA |
| Lowest Common Ancestor of a BST | `trees/lowest_common_ancestor_of_a_bst.py` · `practice/simple/29_lowest_common_ancestor_of_a_bst.py` | walk down while both values are on one side; the first node between them is the LCA |
| Maximum Depth of Binary Tree | `trees/maximum_depth_of_binary_tree.py` | 1 + max(depth(left), depth(right)); `None` is 0 |
| Path Sum | `trees/path_sum.py` | pass the remaining sum down; test it only at a leaf |
| Path Sum II | `trees/path_sum_ii.py` | one shared path: append on entry, pop on exit, copy at a matching leaf |
| Recover a Tree From Preorder Traversal | `trees/recover_a_tree_from_preorder_traversal.py` | stack of ancestors; dash count = depth; pop until len(stack) == depth; fill left before right |
| Recover Binary Search Tree | `trees/recover_binary_search_tree.py` | inorder dips: first = prev of the first dip, second = node of the last dip; swap the values |
| Same Tree | `trees/same_tree.py` | recurse on pairs: both None → True; one None or different values → False |
| Serialize and Deserialize Binary Tree | `trees/serialize_and_deserialize_binary_tree.py` | preorder with `#` for None; the reader replays the walk with one shared iterator |
| Serialize and Deserialize N-ary Tree | `trees/serialize_and_deserialize_n_ary_tree.py` | preorder "value, child count"; the reader builds exactly that many children |
| Subtree of Another Tree | `trees/subtree_of_another_tree.py` | serialize both with null markers and a delimiter; a subtree is a substring |
| Sum of Left Leaves | `trees/sum_of_left_leaves.py` | pass "I am a left child" down; add the leaves that have it |
| Symmetric Tree | `trees/symmetric_tree.py` | mirror(a, b): equal values, mirror(a.left, b.right) and mirror(a.right, b.left) |
| Validate Binary Search Tree | `trees/validate_binary_search_tree.py` · `practice/simple/27_validate_binary_search_tree.py` | pass an open window (low, high) down; left caps high, right raises low |
| Vertical Order Traversal of a Binary Tree | `trees/vertical_order_traversal_of_a_binary_tree.py` | DFS with (row, col); bucket by column; sort each bucket by (row, value) |

### Self-check

1. In Diameter, why is the value you return different from the value you record?
<details><summary>Answer</summary>The parent can extend a path through only one of your arms (a path can't fork), so it needs <code>1 + max(left, right)</code>. The answer, though, may bend at you and use both arms: <code>left + right</code>. Two different quantities, so one is returned and the other recorded.</details>

2. Why isn't "each node is bigger than its left child and smaller than its right child" enough for a BST?
<details><summary>Answer</summary>The rule must hold for whole subtrees: <code>[5, 4, 6, None, None, 3, 7]</code> passes the local check, but 3 sits in 5's right subtree. Passing an open window <code>(low, high)</code> down enforces every ancestor's constraint at once.</details>

3. In Binary Tree Cameras, why does a `None` child report "covered" rather than "needs"?
<details><summary>Answer</summary>If <code>None</code> reported "needs", every leaf would be forced to take a camera. A leaf's camera covers only the leaf and its parent, while the parent's camera covers the leaf, the parent and the grandparent: never worse. Reporting "covered" for <code>None</code> makes leaves report "needs", which pushes each camera one level up.</details>

4. In BFS, why is `len(queue)` at the start of a round exactly one level?
<details><summary>Answer</summary>When a round starts, every node of depth d is in the queue and none of their children has been added yet. Popping exactly that many nodes finishes depth d, while the children line up behind them to form the next round.</details>

<a id="s12"></a>

## Tries

> A trie is a dictionary of dictionaries: each node maps the next letter to a child node, so every word is a path from the root, and words that share a prefix share the path. A lookup costs one step per letter no matter how many words are stored, and one missing letter rules out every word with that prefix at once.

**Reach for it when** there are many words and the questions are about **prefixes** (starts with, autocomplete, the shortest root); when many words must be matched **at the same time** against one board, sentence or stream; or when a search has **wildcards**.

**In this repo:** `tries/` (8 problems) · bank: `practice/simple/31_implement_trie.py` · basics: `practice/simple/basics/tries/` (insert/search/starts-with, delete with pruning, autocomplete, wildcard search).

### The picture

```text
words: app, apple, apply, ape, bat                      * = a word ends here (the end flag)

               (root)
              /      \
             a        b
             |        |
             p        a
           /   \      |
         p*     e*    t*
         |
         l
       /   \
     e*     y*

search("app")         a -> p -> p*          the path exists AND ends on a *   -> True
search("appl")        a -> p -> p -> l      the path exists, but l has no *   -> False (just a prefix)
starts_with("appl")   a -> p -> p -> l      the path exists                   -> True
search("apt")         a -> p -> (no t)      every word starting with "apt" is ruled out in one step
```

Each node is a small object: a dict `children` (letter → node) and a flag `end`. The words never appear as strings; a word is the sequence of letters on the edges from the root to a node with `end = True`.

Why it is fast: the brute force keeps a list of N words and compares the query with each of them: O(N·L) per query (L = word length), re-reading shared prefixes once per word. The trie stores each shared prefix once, so a query walks one path: O(L), whatever N is. The second win is **pruning**: when a letter is missing, the whole subtree (every word with that prefix) is ruled out in one step. That is what makes Word Search II and Stream of Characters fast.

### From idea to code

**The idea in one sentence:** *walk the word down from the root one letter at a time: insert creates missing children, a query follows existing ones and stops at the first missing letter, and the end flag on the last node tells a whole word from a prefix.*

| Decision | Trie answer |
|---|---|
| **State**: what must I remember? | the root node; every node holds `children` (letter → node) and `end` (a word stops here); one finger `node` walks down |
| **Definition**: what exactly does each variable mean? | the node reached by spelling `s` exists ⇔ some inserted word starts with `s`; its `end` ⇔ `s` itself was inserted |
| **Invariant**: what is true at the end of every step? | after reading `word[:i]`, `node` is the node for the prefix `word[:i]` |
| **Step**: how does one letter change the state? | query: `node = node.children.get(ch)` and stop on `None`; insert: create the child if it is missing, then step |
| **Record**: when is the answer updated? | at the last node of an insert: `node.end = True` (or store the word, a count, an index); during a query: note the `end` flags you pass (shortest root, streams) |
| **Init**: starting values | `node = self.root` at the start of every operation |
| **Return**: what comes back? | search: `node is not None and node.end`; starts with: `node is not None`; collect: every word in the subtree below `node` |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "start at the top" | `node = self.root` |
| "follow letter `ch`, if it is there" | `node = node.children.get(ch)` (`None` if not) |
| "make the child if it is missing" | `if ch not in node.children: node.children[ch] = TrieNode()` |
| "a word ends here" | `node.end = True` |
| "a whole word, not just a prefix" | `node is not None and node.end` |
| "every word below this node" | a DFS over `node.children.items()`, adding one letter per level |
| "`.` matches any one letter" | `any(match(child, i + 1) for child in node.children.values())` |
| "rule out every word with this prefix" | `if ch not in node.children: return ...` |

The template. The tags are the seven decisions, and their order matters: `insert` creates a missing child *before* stepping into it, and sets the end flag (RECORD) only after the loop, because only the last node is where the word ends.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}                    # STATE: letter -> TrieNode
        self.end = False                      # STATE: a word stops exactly here


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root                      # INIT: every walk starts at the root
        for ch in word:
            if ch not in node.children:       # create the missing child first ...
                node.children[ch] = TrieNode()
            node = node.children[ch]          # STEP: ... then step down to it
        node.end = True                       # RECORD: mark the word's last node

    def _walk(self, s):                       # the node for prefix s, or None
        node = self.root
        for ch in s:
            node = node.children.get(ch)      # STEP: follow one edge
            if node is None:
                return None                   # no word starts with s
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.end  # RETURN: a whole word, not just a prefix

    def starts_with(self, prefix):
        return self._walk(prefix) is not None # RETURN: the path exists


trie = Trie()
for w in ["app", "apple", "apply", "ape", "bat"]:
    trie.insert(w)
print(trie.search("app"), trie.search("appl"), trie.starts_with("appl"), trie.search("apt"))   # True False True False

**Try it**
- Delete `node.end = True` and rerun: the line prints `False False True False`. `search` now fails for every word (nothing is marked), while `starts_with` still works.
- Insert only `"apple"` into a fresh `Trie()`: `search("app")` is `False` but `starts_with("app")` is `True`. The end flag is the only difference between the two questions.
- Count the nodes: `count = lambda n: 1 + sum(count(c) for c in n.children.values())`, then `count(trie.root)` is 11 (the root plus 10 letters), while the five words have 19 letters in total. Shared prefixes are stored once.
- `trie.insert("")` marks the root itself: afterwards `trie.search("")` is `True`.

### Watch it work

`show` prints the trie as an outline (one letter per line, indented by depth, `*` = end flag). `trace_walk` follows a string down and reports where it stops.

In [ ]:
def show(node, depth=0):                      # the trie as an outline, * = a word ends here
    for ch in sorted(node.children):
        child = node.children[ch]
        print("    " * depth + ch + (" *" if child.end else ""))
        show(child, depth + 1)


def trace_walk(trie, s):
    node, steps = trie.root, []
    for ch in s:
        node = node.children.get(ch)
        steps.append(f"{ch} {'ok' if node else 'missing'}")
        if node is None:
            break
    verdict = "no word starts with it" if node is None else ("a word" if node.end else "only a prefix")
    print(f"{s!r:8} {' -> '.join(steps):30} {verdict}")


trie = Trie()
for w in ["app", "apple", "apply", "ape", "bat"]:
    trie.insert(w)
show(trie.root)
for s in ["app", "appl", "apt", "bat"]:
    trace_walk(trie, s)

**Try it**
- Add `print(len(node.children))` right after the `if node is None: break` lines and run `trace_walk(trie, "apple")`: 1, 2, 1, 2, 0. Each 2 is a fork where words part ways; 0 means no longer word continues.
- In `show`, drop the `sorted(...)`: under `a p` the `p` branch now comes before `e`, because a dict keeps insertion order and `"app"` was inserted before `"ape"`.
- Run `trie.insert("apt")`, then `show(trie.root)` and `trace_walk(trie, "apt")`: a `t *` appears under `a p`, and `'apt'` is now "a word".

### Where it goes wrong

1. **No end flag.** Without `end`, inserting `"apple"` makes `search("app")` succeed: a path existing only means "some word starts with this".
2. **Walking into the end marker.** In the dict-of-dicts shorthand (`node["$"] = True`), the marker sits among the children, so a DFS over the keys must skip `"$"`, or it tries to walk into `True`.
3. **Wildcard lengths.** A pattern matches only if a word ends exactly where the pattern ends: check `node.end` when `i == len(pattern)`, not "some word continues below".
4. **Stopping too late, or too early.** Replace Words wants the *shortest* root: return at the first `end` on the path. Word Search II must keep walking past a found word, because a longer word can continue from there (`"oat"` and `"oath"`).
5. **Emitting a word twice.** In Word Search II the same word can be spelled along two paths: remove it from its node when found.
6. **Not restoring the board.** Mark a cell as visited before recursing and put its letter back afterwards, or later paths see a corrupted board.
7. **Suffix questions on a forward trie.** "Does some word end at the newest letter of the stream?" needs a trie of *reversed* words, walked from the newest letter backwards.
8. **Deleting too much.** Deleting `"app"` must not remove the nodes `"apple"` still uses: prune a node only if it has no children and ends no word.

### Edge cases to say out loud

The empty string (it marks the root) · a word that is a prefix of another (`app`, `apple`) · inserting the same word twice · a query longer than every word · a pattern of only dots, or a dot at either end · deleting a word that isn't there · characters outside a–z (a dict handles any character).

In [ ]:
t = Trie()
assert not t.search("a") and not t.starts_with("a")            # empty trie
t.insert("a")
assert t.search("a") and t.starts_with("") and not t.search("")  # "" is a prefix of every word
t.insert("a")                                                  # a second insert changes nothing
assert len(t.root.children) == 1 and t.search("a")
t.insert("abc")
assert not t.search("ab") and t.starts_with("ab") and not t.starts_with("abcd")
t.insert("Ünï")                                                # any characters: children is a dict
assert t.search("Ünï") and not t.search("Ün")
print("edge cases pass")

**Try it**
- Predict, then run: `Trie().starts_with("")` is `True`, even on an empty trie. The walk takes zero steps and ends on the root, which always exists. Decide whether your problem wants that.
- Predict, then add: `t.insert(""); assert t.search("")`. The root itself becomes the end of a word.
- In a fresh trie insert `"apple"`, then `"app"`: `search("app")` flips from `False` to `True`, and the node count (the `count` lambda from the first Try it) stays 6. `"app"` only needed its end flag set.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Wildcard search** | `.` tries every child: a DFS over `(node, index)` that branches only at dots | 211 |
| **Delete** | clear `end`, then prune, bottom-up, the nodes with no children that end no word | basics |
| **Collect / autocomplete** | walk to the prefix's node, then DFS below it collecting words; or keep ranked candidates in every node | 642 |
| **Board backtracking + trie** | the grid DFS walks down the trie in lockstep; a letter that isn't a child prunes the path | 212 |
| **Shortest prefix** | stop at the first `end` on the path | 648 |
| **Suffixes of a stream** | insert words reversed; walk the newest letters backwards | 1032 |
| **Two-sided query** | insert `suffix + "#" + word` for every suffix; every node stores the best index | 745 |
| **Prefix → candidates** | every node lists the words below it; backtracking asks for the prefix of column k | 425 |

**Wildcards (211).** A letter follows exactly one edge; a `.` fans out to every child. That is a DFS over (trie node, position in the pattern), and it only branches at the dots.

In [ ]:
class WordDictionary:
    def __init__(self):
        self.root = TrieNode()

    def add_word(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.end = True

    def search(self, pattern):
        def match(node, i):                   # can pattern[i:] be spelled below node?
            if i == len(pattern):
                return node.end               # a word must end exactly here
            ch = pattern[i]
            if ch == ".":
                return any(match(child, i + 1) for child in node.children.values())
            child = node.children.get(ch)
            return child is not None and match(child, i + 1)
        return match(self.root, 0)


d = WordDictionary()
for w in ["bad", "dad", "mad"]:
    d.add_word(w)
print(d.search("pad"), d.search("bad"), d.search(".ad"), d.search("b.."), d.search("b."))   # False True True True False

**Try it**
- Change `return node.end` to `return True`: `d.search("b.")` becomes `True` although no two-letter word exists. The pattern ran out in the middle of `"bad"`.
- Put `print(repr(pattern[i:]))` as the first line of `match` and run `d.search(".ad")`: four lines, `'.ad'`, `'ad'`, `'d'`, `''`. `any` stops at the first child that works, so the `d` and `m` branches are never explored.
- Predict before running: `d.search("...")` is `True` and `d.search("....")` is `False`.

**Delete and autocomplete (basics, 642).** Both are a DFS below a node. Delete is bottom-up: each child tells its parent "I am useless now (no children, no word ends here), cut me off". Autocomplete walks down to the prefix's node and collects every word beneath it.

In [ ]:
def delete_word(trie, word):                  # returns True if the word was there
    if not trie.search(word):
        return False
    def remove(node, i):                      # returns True if the parent may cut `node` off
        if i == len(word):
            node.end = False                  # the word no longer ends here
        else:
            ch = word[i]
            if remove(node.children[ch], i + 1):
                del node.children[ch]         # prune the useless child
        return not node.children and not node.end
    remove(trie.root, 0)
    return True


def words_with_prefix(trie, prefix):          # every stored word that starts with prefix
    node = trie._walk(prefix)
    if node is None:
        return []
    found = []
    def collect(node, letters):
        if node.end:
            found.append("".join(letters))    # RECORD: a word ends here
        for ch in sorted(node.children):      # sorted children -> the words come out sorted
            letters.append(ch)
            collect(node.children[ch], letters)
            letters.pop()                     # backtrack
    collect(node, list(prefix))
    return found


t = Trie()
for w in ["car", "card", "care", "cat", "dog"]:
    t.insert(w)
print(words_with_prefix(t, "car"), words_with_prefix(t, "x"))   # ['car', 'card', 'care'] []
print(delete_word(t, "car"), words_with_prefix(t, "ca"))        # True ['card', 'care', 'cat']
print(delete_word(t, "cat"), sorted(t.root.children["c"].children["a"].children))   # True ['r']

**Try it**
- Then run `delete_word(t, "dog")` and print `sorted(t.root.children)`: `['c']`. The whole d-o-g branch is pruned, because no other word needed it.
- Replace `del node.children[ch]` with `pass` and rerun: `cat` is no longer a word, but `t.starts_with("cat")` is still `True`. A dead branch is left behind.
- `delete_word(t, "ca")` returns `False` and changes nothing: `"ca"` is only a prefix, not a stored word.

**Trie + backtracking on a board (212).** Searching the board once per word repeats the same board paths. Instead, walk the board once and the trie in lockstep: a step to a neighbouring cell is allowed only if its letter is a child of the current trie node, so a dead prefix is dropped after one wrong letter, for every word at once. This cell uses the common shorthand where a node is just a `dict` and the key `"$"` plays the end flag; here `"$"` stores the whole word, so a match can read it directly.

```text
 board        words: oath, pea, eat, rain
 o a a n
 e t a e      o -> a -> t -> h   follows the trie o -> a -> t -> h ($ = "oath"): found
 i h k r      o -> e             no "oe" in the trie: dropped after one step
 i f l v
```

In [ ]:
def find_words(board, words):
    root = {}
    for w in words:                           # dict-of-dicts trie; "$" holds the finished word
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    rows, cols, found = len(board), len(board[0]), []

    def dfs(r, c, parent):
        ch = board[r][c]
        node = parent[ch]
        word = node.pop("$", None)            # RECORD once: popping stops duplicates
        if word is not None:
            found.append(word)
        board[r][c] = "#"                     # visited on this path
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] in node:
                dfs(nr, nc, node)             # only letters the trie still allows
        board[r][c] = ch                      # un-mark: other paths may use this cell
        if not node:
            del parent[ch]                    # prune a branch with nothing left to find

    for r in range(rows):
        for c in range(cols):
            if board[r][c] in root:
                dfs(r, c, root)
    return found


board = [list("oaan"), list("etae"), list("ihkr"), list("iflv")]
print(sorted(find_words(board, ["oath", "pea", "eat", "rain"])))   # ['eat', 'oath']

**Try it**
- Replace `node.pop("$", None)` with `node.get("$")` and run `find_words([list("aa")], ["a"])`: `['a', 'a']`. Each cell spells the word again; popping reports it once.
- Delete the un-mark line `board[r][c] = ch` and rerun: only `['oath']`. The cells used by `"oath"` stay `"#"`, and `"eat"` needs the same `t`.
- Add `"oat"` to the word list: both `"oat"` and `"oath"` are found. The search keeps going *through* the node where `"oat"` was harvested.
- Delete the two pruning lines at the end of `dfs`: the answer is the same. Pruning only saves time, by stopping later cells from re-walking finished branches.

**Stop at the first end (648), and walk backwards (1032).** Replace Words walks each word down a trie of roots and stops at the first `end` it meets: the first one is the shortest root. Stream of Characters asks "does some word end at the newest letter?", which is a *prefix* question about the stream read backwards: insert every word reversed and walk the recent letters newest-first.

In [ ]:
def replace_words(roots, sentence):
    trie = Trie()
    for r in roots:
        trie.insert(r)
    def shortest_root(word):
        node = trie.root
        for i, ch in enumerate(word):
            node = node.children.get(ch)
            if node is None:
                return word                   # no root is a prefix: keep the word
            if node.end:
                return word[:i + 1]           # the FIRST end on the path is the shortest root
        return word
    return " ".join(shortest_root(w) for w in sentence.split())


class StreamChecker:
    def __init__(self, words):
        self.trie, self.longest = Trie(), max(map(len, words))
        for w in words:
            self.trie.insert(w[::-1])         # reversed: the newest letter is matched first
        self.recent = deque()

    def query(self, letter):
        self.recent.append(letter)
        if len(self.recent) > self.longest:
            self.recent.popleft()             # older letters can never be part of a match
        node = self.trie.root
        for ch in reversed(self.recent):      # newest -> oldest
            node = node.children.get(ch)
            if node is None:
                return False
            if node.end:
                return True                   # some word ends at the newest letter
        return False


print(replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery"))   # the cat was rat by the bat
sc = StreamChecker(["cd", "f", "kl"])
print([ch for ch in "abcdefghijkl" if sc.query(ch)])                                  # ['d', 'f', 'l']

**Try it**
- Add the root `"ca"` and predict the first replacement: `"the ca was rat by the bat"`. The shorter root is met first on the path.
- Insert the words forward (`self.trie.insert(w)`) and rerun: only `['f']`. A one-letter word reads the same in both directions; `"cd"` and `"kl"` are now looked for backwards.
- Delete the two `popleft` lines: the answers don't change, and the walk stays short (it can't go deeper than the longest word in the trie). Only the memory now grows with the stream.

**One prefix for two constraints (745), and a trie that feeds backtracking (425).** "Starts with `pref` and ends with `suff`" becomes a single prefix question if you insert, for every suffix of every word, the string `suffix + "#" + word`; every node on the way remembers the latest (largest) index that passed through it. Word Squares uses a trie the other way round: a backtracking search fills the square row by row, and row k must start with column k of the rows above, so the trie answers "which words start with this prefix?".

```text
 "apple" (index 0) is inserted 6 times:  "#apple", "e#apple", "le#apple", "ple#apple", ...
 f(pref = "ap", suff = "le")  ->  walk "le#ap"  ->  the index stored on that node: 0

 word square: row k = column k        w a l l
                                      a r e a       row 2 must start with "le":
                                      l e a d       column 2 of the rows "wall", "area"
                                      l a d y
```

In [ ]:
class WordFilter:
    def __init__(self, words):
        self.root = {}
        for index, word in enumerate(words):         # increasing index: a later word overwrites
            for k in range(len(word) + 1):           # every suffix, including the empty one
                node = self.root
                for ch in word[k:] + "#" + word:
                    node = node.setdefault(ch, {})
                    node["$"] = index                # every node on the path knows the best index

    def f(self, pref, suff):
        node = self.root
        for ch in suff + "#" + pref:
            if ch not in node:
                return -1
            node = node[ch]
        return node["$"]


def word_squares(words):
    n = len(words[0])
    root = {"$": list(words)}                        # every node lists the words below it
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {"$": []})
            node["$"].append(w)

    def starting_with(prefix):
        node = root
        for ch in prefix:
            if ch not in node:
                return []
            node = node[ch]
        return node["$"]

    squares, square = [], []
    def fill():
        k = len(square)
        if k == n:
            squares.append(square[:])
            return
        prefix = "".join(row[k] for row in square)   # column k of the rows so far
        for w in starting_with(prefix):
            square.append(w)
            fill()
            square.pop()
    fill()
    return squares


wf = WordFilter(["apple", "ample", "apply"])
print(wf.f("ap", "le"), wf.f("a", "e"), wf.f("b", ""))   # 0 1 -1
squares = word_squares(["area", "lead", "wall", "lady", "ball"])
print(len(squares), [sq[0] for sq in squares])          # 2 ['wall', 'ball']
for row in squares[0]:
    print(" ".join(row))                                # w a l l / a r e a / l e a d / l a d y

**Try it**
- `WordFilter(["ab", "ab"]).f("ab", "")` is 1: the later duplicate overwrote the index on every node of its path.
- Skip the empty suffix (`range(len(word))`): `wf.f("ap", "")` becomes -1 although `"apply"` starts with `"ap"`. Every query with `suff = ""` walks `"#..."`, and no key starts with `"#"` any more.
- Print `k, prefix, starting_with(prefix)` right after `prefix` is computed in `fill`: the prefixes `'r'`, `'e'` and `'de'` die at once, because no word starts with them.

### Say it in the interview

> "Checking the query against every word costs O(N·L) per query and re-reads shared prefixes. I'll store the words in a trie: one node per prefix, children in a dict, and an end flag. A query walks one path, O(L), and a missing letter rules out every word with that prefix at once. Space is O(total letters)."

While coding, point at the end flag ("this separates a word from a prefix") and at the early `return` on a missing child ("this is the pruning"). For board or stream problems, say what the trie lets you do *simultaneously*: "every word is checked at once, by one walk".

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Design Add and Search Words Data Structure | `tries/design_add_and_search_words_data_structure.py` | a letter follows one edge, `.` tries every child; match only if a word ends exactly where the pattern ends |
| Design Search Autocomplete System | `tries/design_search_autocomplete_system.py` | a cursor moves one edge per keystroke; each node keeps sentence counts; top 3 by (−count, sentence) |
| Implement Trie (Prefix Tree) | `tries/implement_trie_prefix_tree.py` · `practice/simple/31_implement_trie.py` | dict of children + end flag; search needs the flag, startsWith only the path |
| Prefix and Suffix Search | `tries/prefix_and_suffix_search.py` | insert `suffix#word` for every suffix; nodes store the latest index; query `suff#pref` |
| Replace Words | `tries/replace_words.py` | walk each word down the trie of roots; the first end flag is the shortest root |
| Stream of Characters | `tries/stream_of_characters.py` | trie of reversed words; walk the last L letters newest-first |
| Word Search II | `tries/word_search_ii.py` | the board DFS moves down the trie in lockstep; pop found words, prune empty branches |
| Word Squares | `tries/word_squares.py` | row k must start with column k so far; trie nodes list the words with each prefix |

### Self-check

1. After inserting only `"apple"`, why is `search("app")` false but `starts_with("app")` true?
<details><summary>Answer</summary>Both walks reach the node for "app", because "apple" passes through it. <code>starts_with</code> only needs the path to exist; <code>search</code> also needs the end flag on that node, and only the node for "apple" has it.</details>

2. In Word Search II, why pop the word from its node instead of collecting matches into a set?
<details><summary>Answer</summary>Popping reports each word exactly once, and it lets a branch become empty. Empty branches are then deleted, so later starting cells never re-walk prefixes whose words have all been found. A set would remove the duplicates but keep all that wasted walking.</details>

3. Why does Stream of Characters insert the words reversed?
<details><summary>Answer</summary>The question is whether a word ends at the newest letter, so the word's <em>last</em> letter must be matched first. Reading the stream backwards from the newest letter turns "suffix of the stream" into "prefix of the reversed stream", which a trie of reversed words answers with one walk.</details>

4. Why must WordFilter also insert the empty suffix, `"#" + word`?
<details><summary>Answer</summary>A query with <code>suff = ""</code> walks <code>"#" + pref</code>. That path exists only if some key starts with <code>"#"</code>, which is exactly the empty-suffix key. Without it, every prefix-only query returns -1.</details>